# 14 — Cost, Operating Points, Fine-Tune Update Size, and Durability to Unseen Attacks

Four additions on the same corpora, caps and splits as notebooks 10 to 13, all at the reference seed.

Cost. Every strategy that fits a model already carries a fitting time in the earlier result files; the post-hoc strategies do not, because they fit nothing. This notebook times the calibration fit, the exact threshold sweep and their application, so that the cost comparison the paper is framed on can be reported rather than asserted.

Operating points. MCC and false-positive rate have been reported separately. Here the two are joined in the form an operator chooses on: true-positive rate at a capped false-positive rate, at caps of 1% and 0.1%, with the threshold set two ways. The oracle point uses evaluation labels and bounds what the scores could deliver; the deployable point sets the threshold on the labelled buffer alone and reports what is realised on evaluation data.

Fine-tune update size. Notebook 13 continued the source model with a fixed small update, which makes the scheme conservative by construction. The update is swept here at three sizes for each family: 100, 300 and 1,000 further trees for Random Forest, the same for LightGBM boosting rounds, and 30, 90 and 300 further epochs for the MLP, the middle value matching the source model's own capacity.

Durability to unseen attacks. NF-v2 carries no timestamps, so a temporal holdout is not available. The proxy used here is an attack-family holdout: the buffer is drawn only from families other than one held out, the retrained model is evaluated on the full target and separately on the held-out family's rows, which measures how a model retrained on a small sample behaves when the target later presents an attack type the sample did not contain.

Results append to fc_results_v5.csv with resume-skip on (seed, source, target, model, budget, strategy).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, gc, time, json, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

BASE   = '/content/drive/MyDrive/drift-conference'
CACHE  = f'{BASE}/data/nfv2/cache'
RESULT = f'{BASE}/results/fourcorpus'

CFG = dict(
    corpora       = ['nf2018v2', 'nfunswv2', 'nftonv2', 'nfbotv2'],
    seed          = 42,
    test_size     = 0.30,
    train_cap     = 250_000,
    eval_cap      = 200_000,
    pool_cap      = 200_000,
    budgets       = [0.0001, 0.001, 0.01],
    div_budgets   = [0.0001, 0.001],
    ft_budgets    = [0.001, 0.01],
    hold_budget   = 0.001,
    fpr_caps      = [0.01, 0.001],
    ft_mult       = [1, 3, 10],
    rf_estimators = 300,
    ece_bins      = 15,
    mlp_hidden    = (128, 64),
    mlp_max_iter  = 100,
    max_families  = 3,
    hold_min_rows = 2_000,
    hold_max_share= 0.60,
)
MODELS = ['rf', 'lgbm', 'mlp']
V5_CSV = f'{RESULT}/fc_results_v5.csv'
COLS = ['seed', 'source', 'target', 'model', 'budget', 'strategy', 'n_train', 'held_family',
        'mcc', 'macro_f1', 'fp_rate', 'auprc_macro',
        'tpr_at_fpr01_oracle', 'tpr_at_fpr001_oracle',
        'tpr_at_fpr01_buffer', 'fpr_at_fpr01_buffer',
        'tpr_at_fpr001_buffer', 'fpr_at_fpr001_buffer',
        'tpr_heldout_family', 'fit_s', 'apply_s']
print(json.dumps({k: str(v) for k, v in CFG.items()}, indent=2))

In [ ]:
DATASETS = {tag: pd.read_parquet(f'{CACHE}/{tag}_prepared.parquet') for tag in CFG['corpora']}
FEATURES = [c for c in DATASETS['nf2018v2'].columns if c not in ('Label', 'Attack')]
for tag, d in DATASETS.items():
    assert [c for c in d.columns if c not in ('Label', 'Attack')] == FEATURES, tag
print('features:', len(FEATURES))

In [ ]:
import numpy as np, pandas as pd
from sklearn.metrics import (f1_score, matthews_corrcoef, average_precision_score,
                              brier_score_loss, confusion_matrix)
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb

F32_SAFE = 1e37

def ece_score(y_true, p_pos, n_bins=15):
    conf = np.maximum(p_pos, 1 - p_pos)
    correct = ((p_pos >= 0.5).astype(int) == y_true).astype(float)
    bins = np.linspace(0.5, 1.0, n_bins + 1)
    idx = np.clip(np.digitize(conf, bins) - 1, 0, n_bins - 1)
    ece = 0.0
    for b in range(n_bins):
        m = idx == b
        if m.any():
            ece += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return ece

def mcc_from_counts(tp, fp, fn, tn):
    num = tp * tn - fp * fn
    den = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    with np.errstate(invalid='ignore', divide='ignore'):
        out = np.where(den > 0, num / den, 0.0)
    return out

def best_threshold_mcc(y_true, p_pos, n_grid=199):
    """Max MCC over a quantile grid of thresholds; robust to degenerate score distributions."""
    y = np.asarray(y_true).astype(int); p = np.asarray(p_pos)
    thr = np.unique(np.quantile(p, np.linspace(0.001, 0.999, n_grid)))
    if len(thr) < 2:
        thr = np.array([thr[0]]) if len(thr) else np.array([0.5])
    P = y.sum(); N = len(y) - P
    tp = np.array([(y[p >= t]).sum() for t in thr], dtype=float)
    fp = np.array([(p >= t).sum() for t in thr], dtype=float) - tp
    fn = P - tp; tn = N - fp
    mccs = mcc_from_counts(tp, fp, fn, tn)
    i = int(np.argmax(mccs))
    return float(mccs[i]), float(thr[i])

def all_metrics(y_true, p_pos, ece_bins=CFG['ece_bins']):
    y_true = np.asarray(y_true); p_pos = np.asarray(p_pos)
    pred = (p_pos >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    ap_att = average_precision_score(y_true, p_pos)
    ap_ben = average_precision_score(1 - y_true, 1 - p_pos)
    mb, tb = best_threshold_mcc(y_true, p_pos)
    return dict(
        macro_f1=f1_score(y_true, pred, average='macro'),
        weighted_f1=f1_score(y_true, pred, average='weighted'),
        mcc=matthews_corrcoef(y_true, pred),
        auprc_macro=(ap_att + ap_ben) / 2,
        fp_rate=fp / (fp + tn) if (fp + tn) else np.nan,
        brier=brier_score_loss(y_true, p_pos),
        ece=ece_score(y_true, p_pos, ece_bins),
        mcc_best_thr=mb, thr_best=tb,
    )

def clean_X(df, features, medians=None):
    X = df[features].astype('float64')
    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.mask(X.abs() > F32_SAFE, np.nan)
    if medians is None:
        medians = X.median()
    return X.fillna(medians), medians

def make_model(name, seed, n_rows, rf_estimators=CFG['rf_estimators'], mlp_hidden=CFG['mlp_hidden'], mlp_max_iter=CFG['mlp_max_iter']):
    if name == 'rf':
        return RandomForestClassifier(n_estimators=rf_estimators, n_jobs=-1, random_state=seed)
    if name == 'lgbm':
        return lgb.LGBMClassifier(n_estimators=rf_estimators, random_state=seed, n_jobs=-1, verbosity=-1)
    # early stopping needs a stratifiable validation split; disable on tiny buffers
    small = n_rows < 5000
    return Pipeline([
        ('scaler', StandardScaler()),
        ('clf', MLPClassifier(hidden_layer_sizes=mlp_hidden, activation='relu', solver='adam',
                              batch_size=min(1024, max(8, n_rows // 4)),
                              max_iter=(200 if small else mlp_max_iter),
                              early_stopping=not small, validation_fraction=0.05,
                              n_iter_no_change=5, random_state=seed)),
    ])

def platt_fit(scores, y):
    y = np.asarray(y)
    if len(np.unique(y)) < 2:
        return None
    lr = LogisticRegression(max_iter=1000)
    lr.fit(np.asarray(scores).reshape(-1, 1), y)
    return lr

def platt_apply(lr, scores, y_buf):
    if lr is None:
        return np.full(len(scores), float(np.asarray(y_buf)[0]))
    return lr.predict_proba(np.asarray(scores).reshape(-1, 1))[:, 1]

def stratified_frac(df, frac, seed, min_per_group=1):
    """Per-family sample of round(n*frac) rows, floored at min_per_group; no groupby.apply."""
    rng_state = seed
    parts = []
    for fam, g in df.groupby('Attack', sort=True):
        n = min(len(g), max(min_per_group, int(round(len(g) * frac))))
        parts.append(g.sample(n=n, random_state=rng_state))
    return pd.concat(parts).sample(frac=1.0, random_state=seed).reset_index(drop=True)

def stratified_cap(df, cap, seed):
    if len(df) <= cap:
        return df.reset_index(drop=True)
    return stratified_frac(df, cap / len(df), seed)

In [ ]:
import numpy as np, pandas as pd
from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import matthews_corrcoef

def mcc_from_counts(tp, fp, fn, tn):
    num = tp * tn - fp * fn
    den = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    with np.errstate(invalid='ignore', divide='ignore'):
        return np.where(den > 0, num / den, 0.0)

def best_threshold_mcc(y_true, p_pos):
    y = np.asarray(y_true).astype(np.int64); p = np.asarray(p_pos, dtype=np.float64)
    order = np.argsort(-p, kind='mergesort'); ps, ys = p[order], y[order]
    P = int(ys.sum()); N = len(ys) - P
    tp = np.cumsum(ys); fp = np.cumsum(1 - ys)
    last = np.r_[ps[1:] != ps[:-1], True]
    tp, fp, cuts = tp[last].astype(float), fp[last].astype(float), ps[last]
    mccs = mcc_from_counts(tp, fp, P - tp, N - fp)
    i = int(np.argmax(mccs))
    return (0.0, float('inf')) if mccs[i] <= 0 else (float(mccs[i]), float(cuts[i]))

def best_threshold_two_sided(y_true, p_pos):
    """Max MCC over both orientations. Returns (mcc, thr, orient) with orient +1 (p>=thr) or -1 ((1-p)>=thr)."""
    m_pos, t_pos = best_threshold_mcc(y_true, p_pos)
    m_neg, t_neg = best_threshold_mcc(y_true, 1.0 - np.asarray(p_pos))
    return (m_pos, t_pos, 1) if m_pos >= m_neg else (m_neg, t_neg, -1)

def apply_oriented(p_pos, thr, orient):
    p = np.asarray(p_pos)
    return ((p if orient == 1 else 1.0 - p) >= thr).astype(int)

def mcc_of_pred(y_true, pred):
    return matthews_corrcoef(np.asarray(y_true), np.asarray(pred))

# ---------- acquisition rules: return integer positions into the pool ----------
def acquire_uniform(n_pool, k, seed):
    return np.random.default_rng(seed).choice(n_pool, size=min(k, n_pool), replace=False)

def acquire_uncertainty(p_pool, k):
    p = np.clip(np.asarray(p_pool), 1e-9, 1 - 1e-9)
    ent = -(p * np.log(p) + (1 - p) * np.log(1 - p))
    return np.argsort(-ent, kind='mergesort')[:k]

def acquire_diversity(X_pool, k, seed):
    """k-means with k clusters on standardised features; per cluster, the member nearest its centroid.
    Distances are computed to each row's own centroid only (O(n*features)), never as an n x k matrix."""
    Xs = StandardScaler().fit_transform(np.asarray(X_pool, dtype=np.float64))
    k = min(k, len(Xs))
    km = MiniBatchKMeans(n_clusters=k, random_state=seed, batch_size=4096, n_init=1, max_iter=50).fit(Xs)
    labels = km.labels_
    own = np.einsum('ij,ij->i', Xs - km.cluster_centers_[labels], Xs - km.cluster_centers_[labels])
    df = pd.DataFrame({'lab': labels, 'd': own})
    chosen = df.groupby('lab').d.idxmin().values.astype(int)
    if len(chosen) < k:
        rest = np.setdiff1d(np.arange(len(Xs)), chosen)
        chosen = np.concatenate([chosen, rest[np.argsort(own[rest])[:k - len(chosen)]]])
    return chosen

def acquire_hybrid(X_pool, p_pool, k, seed, factor=5):
    cand = acquire_uncertainty(p_pool, min(len(p_pool), factor * k))
    sub = acquire_diversity(np.asarray(X_pool)[cand], k, seed)
    return cand[sub]

# ---------- cross-validated buffer-only estimate on the buffer itself ----------
def cv_estimate(make_model_fn, Xb, yb, seed, n_splits=3):
    """Mean MCC over stratified folds; returns 0.0 when a class has fewer than n_splits rows."""
    yb = np.asarray(yb)
    if len(np.unique(yb)) < 2 or np.bincount(yb).min() < n_splits:
        return 0.0
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    out = []
    for tr, te in skf.split(Xb, yb):
        if len(np.unique(yb[tr])) < 2:
            out.append(0.0); continue
        mdl = make_model_fn(len(tr)); mdl.fit(Xb.iloc[tr], yb[tr])
        out.append(mcc_of_pred(yb[te], (mdl.predict_proba(Xb.iloc[te])[:, 1] >= 0.5).astype(int)))
    return float(np.mean(out))

In [ ]:
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import matthews_corrcoef
import lightgbm as lgb

# ---------- fine-tuning: keep the source model, adapt it on the buffer ----------
def finetune(model, name, Xb, yb, seed, add_trees=100, ft_epochs=30):
    """Family-specific continuation of a fitted source model on the buffer.
    RF: warm_start adds `add_trees` grown on the buffer, source trees retained.
    LightGBM: boosting continues from the source booster via init_model.
    MLP: SGD continues from the source weights, with the buffer transformed by the
    SOURCE scaler; refitting the pipeline would refit the scaler on the buffer and
    apply the source weights to differently-scaled inputs.
    A single-class buffer offers nothing to adapt to, so the source model is returned."""
    yb = np.asarray(yb)
    if len(np.unique(yb)) < 2:
        return model
    if name == 'rf':
        model.set_params(warm_start=True, n_estimators=model.n_estimators + add_trees)
        model.fit(Xb, yb)
        return model
    if name == 'lgbm':
        m2 = lgb.LGBMClassifier(n_estimators=add_trees, random_state=seed, n_jobs=-1, verbosity=-1)
        m2.fit(Xb, yb, init_model=model.booster_)
        return m2
    scaler = model.named_steps['scaler']
    clf = model.named_steps['clf']
    # sklearn leaves best_loss_ unset when the source fit used early stopping; the
    # non-early-stopping update path dereferences it, so it is restored here.
    if getattr(clf, 'best_loss_', None) is None:
        curve = getattr(clf, 'loss_curve_', None)
        clf.best_loss_ = float(curve[-1]) if curve else np.inf
    clf._no_improvement_count = 0
    clf.set_params(warm_start=True, max_iter=ft_epochs, early_stopping=False)
    clf.fit(scaler.transform(Xb), yb)
    return model


# ---------- importance weighting: reweight source rows towards the target ----------
def importance_weights(Xs, Xb, seed, clip=(0.05, 20.0), n_src=20000):
    """Domain discriminator p(target|x); source weight = p/(1-p).
    The discriminator is trained on a class-balanced sample and with balanced class
    weights, because an unbalanced source/buffer ratio otherwise drives every source
    probability to zero and collapses the weights onto the clip floor."""
    rng = np.random.default_rng(seed)
    n = min(n_src, len(Xs), max(len(Xb), 200))
    idx = rng.choice(len(Xs), size=min(n_src, len(Xs)), replace=False)
    src_d = np.asarray(Xs)[rng.choice(len(Xs), size=n, replace=False)]
    tgt_d = np.asarray(Xb)
    if len(tgt_d) < n:
        tgt_d = tgt_d[rng.choice(len(tgt_d), size=n, replace=True)]
    Xd = np.vstack([src_d, tgt_d])
    yd = np.r_[np.zeros(len(src_d)), np.ones(len(tgt_d))]
    mu, sd = Xd.mean(0), Xd.std(0) + 1e-9
    lr = LogisticRegression(max_iter=300, C=1.0, class_weight='balanced')
    lr.fit((Xd - mu) / sd, yd)
    p = lr.predict_proba((np.asarray(Xs) - mu) / sd)[:, 1]
    w = p / np.clip(1 - p, 1e-6, None)
    w = np.clip(w, *clip)
    return w / w.mean()


# ---------- selector estimator that does not collapse on tiny buffers ----------
def cv_estimate_v2(make_model_fn, Xb, yb, seed):
    """Stratified CV with folds adapted to the rarest class; falls back to a
    single stratified holdout when the minority class has 2 rows, and to the
    buffer-resubstitution estimate when it has 1. Never returns 0 by default."""
    yb = np.asarray(yb)
    if len(np.unique(yb)) < 2:
        return 0.0, 'single_class'
    nmin = int(np.bincount(yb).min())
    if nmin >= 3:
        k = min(3, nmin)
        skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=seed)
        out = []
        for tr, te in skf.split(Xb, yb):
            if len(np.unique(yb[tr])) < 2:
                continue
            m = make_model_fn(len(tr)); m.fit(Xb.iloc[tr], yb[tr])
            out.append(matthews_corrcoef(yb[te], (m.predict_proba(Xb.iloc[te])[:, 1] >= 0.5).astype(int)))
        return (float(np.mean(out)), f'cv{k}') if out else (0.0, 'cv_empty')
    if nmin == 2:
        skf = StratifiedKFold(n_splits=2, shuffle=True, random_state=seed)
        out = []
        for tr, te in skf.split(Xb, yb):
            if len(np.unique(yb[tr])) < 2:
                continue
            m = make_model_fn(len(tr)); m.fit(Xb.iloc[tr], yb[tr])
            out.append(matthews_corrcoef(yb[te], (m.predict_proba(Xb.iloc[te])[:, 1] >= 0.5).astype(int)))
        if out and max(out) > 0:
            return float(np.mean(out)), 'holdout2'
        m = make_model_fn(len(Xb)); m.fit(Xb, yb)
        return float(matthews_corrcoef(yb, (m.predict_proba(Xb)[:, 1] >= 0.5).astype(int))), 'resub2'
    m = make_model_fn(len(Xb)); m.fit(Xb, yb)
    return float(matthews_corrcoef(yb, (m.predict_proba(Xb)[:, 1] >= 0.5).astype(int))), 'resub'

# ---------- iterative active learning: the retrained model drives the queries ----------
def entropy(p):
    p = np.clip(np.asarray(p), 1e-9, 1 - 1e-9)
    return -(p * np.log(p) + (1 - p) * np.log(1 - p))

def al_iterative(make_model_fn, pool, features, clean_fn, k_total, rounds, seed, stratify_col='Attack'):
    """Round 1 is a stratified random seed set; later rounds query the highest-entropy
    pool rows under the model retrained on what has been labelled so far."""
    per = max(1, k_total // rounds)
    rng_state = seed
    first = pool.groupby(stratify_col, group_keys=False).apply(
        lambda g: g.sample(n=max(1, int(round(len(g) * per / len(pool)))), random_state=rng_state))
    chosen = list(first.index[:per])
    for _ in range(rounds - 1):
        lab = pool.loc[chosen]
        Xl, med = clean_fn(lab, features)
        yl = lab['Label'].values
        if len(np.unique(yl)) < 2:
            rest = pool.index.difference(chosen)
            chosen += list(pd.Index(rest)[:per]); continue
        m = make_model_fn(len(Xl)); m.fit(Xl, yl)
        rest = pool.index.difference(chosen)
        Xr, _ = clean_fn(pool.loc[rest], features, medians=med)
        e = entropy(m.predict_proba(Xr)[:, 1])
        chosen += list(pd.Index(rest)[np.argsort(-e)[:per]])
    return pool.loc[chosen[:k_total]]

# ---------- INSOMNIA-style: pseudo-labelled pool plus uncertainty-queried labels ----------
def insomnia_buffer(source_model, pool, features, clean_fn, med_s, k, seed, conf=0.9, cap=50000):
    """Approximates INSOMNIA: the frozen source model pseudo-labels confident pool rows,
    and the analyst budget is spent on the rows it is least certain about."""
    Xp, _ = clean_fn(pool, features, medians=med_s)
    p = source_model.predict_proba(Xp)[:, 1]
    q = pool.index[np.argsort(-entropy(p))[:k]]
    conf_mask = (p >= conf) | (p <= 1 - conf)
    conf_idx = pool.index[conf_mask].difference(q)
    if len(conf_idx) > cap:
        conf_idx = pd.Index(np.random.default_rng(seed).choice(conf_idx, cap, replace=False))
    pseudo = pool.loc[conf_idx].copy()
    pseudo['Label'] = (p[pool.index.get_indexer(conf_idx)] >= 0.5).astype(int)
    return pool.loc[q], pseudo

In [ ]:
import numpy as np, pandas as pd

def tpr_at_fpr(y, p, target_fpr):
    """Highest TPR attainable at or below target_fpr, with the threshold that attains it.
    Thresholds are evaluated at every distinct score, so the result is exact."""
    y = np.asarray(y).astype(int); p = np.asarray(p, dtype=float)
    order = np.argsort(-p, kind='mergesort'); ys, ps = y[order], p[order]
    P = int(ys.sum()); N = len(ys) - P
    if P == 0 or N == 0:
        return np.nan, np.nan
    tp = np.cumsum(ys); fp = np.cumsum(1 - ys)
    last = np.r_[ps[1:] != ps[:-1], True]
    tp, fp, cuts = tp[last], fp[last], ps[last]
    ok = (fp / N) <= target_fpr
    if not ok.any():
        return 0.0, float(cuts[0]) + 1e-12
    i = int(np.argmax(np.where(ok, tp, -1)))
    return float(tp[i] / P), float(cuts[i])

def threshold_for_fpr_on_buffer(y_buf, p_buf, target_fpr):
    """Operating point an analyst could actually set: chosen on the labelled buffer only."""
    _, thr = tpr_at_fpr(y_buf, p_buf, target_fpr)
    return thr

def realised_at_threshold(y, p, thr):
    """TPR and FPR on the evaluation set at an externally chosen threshold."""
    y = np.asarray(y).astype(int); pred = (np.asarray(p) >= thr).astype(int)
    P = int(y.sum()); N = len(y) - P
    tp = int(((pred == 1) & (y == 1)).sum()); fp = int(((pred == 1) & (y == 0)).sum())
    return (tp / P if P else np.nan), (fp / N if N else np.nan)

def family_holdout_buffer(train_full, held_family, frac, seed, min_per_group=1):
    """Stratified buffer drawn only from families other than held_family, so the retrained
    model has never seen that attack type. Size matches the ordinary buffer at this budget."""
    pool = train_full[train_full['Attack'] != held_family]
    k = max(1, int(round(len(train_full) * frac)))
    parts = []
    for fam, g in pool.groupby('Attack', sort=True):
        n = min(len(g), max(min_per_group, int(round(len(g) * k / len(pool)))))
        parts.append(g.sample(n=n, random_state=seed))
    return pd.concat(parts).sample(frac=1.0, random_state=seed).reset_index(drop=True)

def eligible_families(df, min_rows=2000, max_share=0.60):
    """Attack families large enough to matter but not so dominant that holding one out
    leaves nothing to train on."""
    v = df.loc[df['Attack'] != 'Benign', 'Attack'].value_counts()
    n_att = int((df['Attack'] != 'Benign').sum())
    return [f for f, c in v.items() if c >= min_rows and c / n_att <= max_share]

def metrics_at_threshold(y_true, p_pos, thr):
    """Threshold-sensitive metrics taken at an externally chosen cut rather than at 0.5.
    Needed because the rethreshold strategy does not operate at 0.5, so reporting its
    macro-F1 or false-positive rate from the default cut would describe a different detector."""
    from sklearn.metrics import f1_score, matthews_corrcoef, confusion_matrix
    y = np.asarray(y_true).astype(int)
    pred = (np.asarray(p_pos) >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return dict(mcc=matthews_corrcoef(y, pred),
                macro_f1=f1_score(y, pred, average='macro'),
                fp_rate=fp / (fp + tn) if (fp + tn) else np.nan)

In [ ]:
from sklearn.model_selection import train_test_split
import copy

def record(row):
    r = {c: row.get(c, np.nan) for c in COLS}
    pd.DataFrame([r], columns=COLS).to_csv(V5_CSV, mode='a', index=False, header=not os.path.exists(V5_CSV))

def key(src, tgt, m, b, s, fam=''):
    return (src, tgt, m, f'{float(b):.6g}', s, str(fam))

done = set()
if os.path.exists(V5_CSV):
    prev = pd.read_csv(V5_CSV)
    done = set(key(r.source, r.target, r.model, r.budget, r.strategy,
                   '' if pd.isna(r.held_family) else r.held_family) for r in prev.itertuples())
    print(f'resume: {len(done)} rows already recorded')

def is_done(*k):
    return key(*k) in done

def operating_points(yev, p_ev, ybuf, p_buf):
    out = {}
    for cap, tag in zip(CFG['fpr_caps'], ['01', '001']):
        t_or, _ = tpr_at_fpr(yev, p_ev, cap)
        out[f'tpr_at_fpr{tag}_oracle'] = t_or
        thr = threshold_for_fpr_on_buffer(ybuf, p_buf, cap)
        if np.isnan(thr):
            out[f'tpr_at_fpr{tag}_buffer'] = np.nan; out[f'fpr_at_fpr{tag}_buffer'] = np.nan
        else:
            rt, rf = realised_at_threshold(yev, p_ev, thr)
            out[f'tpr_at_fpr{tag}_buffer'] = rt; out[f'fpr_at_fpr{tag}_buffer'] = rf
    return out

def mark(src, tgt, m, b, s, metrics, n_train, fam='', **extra):
    record(dict(seed=CFG['seed'], source=src, target=tgt, model=m, budget=b, strategy=s,
                n_train=n_train, held_family=fam, **metrics, **extra))
    done.add(key(src, tgt, m, b, s, fam))
    print(f"  {src}->{tgt} {m} b={b} {s}{('/'+str(fam)) if fam else ''}: "
          f"MCC={metrics.get('mcc', float('nan')):.3f} TPR@1%={metrics.get('tpr_at_fpr01_buffer', float('nan')):.3f}")

seed = CFG['seed']
T0 = time.time()
def el():
    return f'[{(time.time()-T0)/60:5.1f}m]'

parts = {}
for tag, d in DATASETS.items():
    tr, te = train_test_split(d, test_size=CFG['test_size'], stratify=d['Attack'], random_state=seed)
    tr = tr.reset_index(drop=True)
    parts[tag] = dict(train_full=tr, train=stratified_cap(tr, CFG['train_cap'], seed),
                      eval=stratified_cap(te, CFG['eval_cap'], seed).reset_index(drop=True),
                      pool=stratified_cap(tr, CFG['pool_cap'], seed).reset_index(drop=True))

# Buffers depend only on (target, budget) and the seed, so they are built once here rather than
# rebuilt inside the source loop, where each rebuild is a groupby over a 2.1M-row partition.
# Diversity is restricted to the budgets notebook 12 used, both for comparability and because
# its clustering cost grows with k and is prohibitive at the largest budget.
BUF, DIV, HOLD, FAMS = {}, {}, {}, {}
for tgt in CFG['corpora']:
    for b in CFG['budgets']:
        BUF[(tgt, b)] = stratified_frac(parts[tgt]['train_full'], b, seed)
    Xpool_raw, _ = clean_X(parts[tgt]['pool'], FEATURES)
    for b in CFG['div_budgets']:
        k = max(1, int(round(len(parts[tgt]['train_full']) * b)))
        t0 = time.time()
        DIV[(tgt, b)] = parts[tgt]['pool'].iloc[acquire_diversity(Xpool_raw.values, k, seed)]
        print(f'{el()} diversity buffer {tgt} b={b} k={k}: {time.time()-t0:.0f}s')
    del Xpool_raw; gc.collect()
    FAMS[tgt] = eligible_families(parts[tgt]['train_full'], CFG['hold_min_rows'], CFG['hold_max_share'])[:CFG['max_families']]
    for fam in FAMS[tgt]:
        HOLD[(tgt, fam)] = family_holdout_buffer(parts[tgt]['train_full'], fam, CFG['hold_budget'], seed)
    print(f'{el()} {tgt}: holdout families {FAMS[tgt]}')

for src in CFG['corpora']:
    others = [t for t in CFG['corpora'] if t != src]
    Xs, med_s = clean_X(parts[src]['train'], FEATURES)
    ys = parts[src]['train']['Label'].values
    for mname in MODELS:
        need = any(not is_done(src, t, mname, b, s)
                   for t in others for b in CFG['budgets'] for s in ('calibrate_cost', 'rethreshold_cost')) or                any(not is_done(src, t, mname, b, f'finetune_x{k}')
                   for t in others for b in CFG['ft_budgets'] for k in CFG['ft_mult'])
        if not need:
            continue
        src_model = make_model(mname, seed, len(Xs))
        t0 = time.time(); src_model.fit(Xs, ys); src_fit = round(time.time() - t0, 1)
        print(f'{el()} source fit {mname} on {src}: {src_fit}s')
        for tgt in others:
            ev = parts[tgt]['eval']; yev = ev['Label'].values
            Xev_s, _ = clean_X(ev, FEATURES, medians=med_s)
            p_ev = src_model.predict_proba(Xev_s)[:, 1]
            for b in CFG['budgets']:
                buf = BUF[(tgt, b)]; ybuf = buf['Label'].values
                Xb_s, _ = clean_X(buf, FEATURES, medians=med_s)
                p_buf = src_model.predict_proba(Xb_s)[:, 1]
                single = len(np.unique(ybuf)) < 2

                if not is_done(src, tgt, mname, b, 'calibrate_cost'):
                    t0 = time.time(); lr = platt_fit(p_buf, ybuf); cal_fit = time.time() - t0
                    t0 = time.time(); p_cal = platt_apply(lr, p_ev, ybuf); cal_ap = time.time() - t0
                    m = all_metrics(yev, p_cal)
                    m.update(operating_points(yev, p_cal, ybuf, platt_apply(lr, p_buf, ybuf)))
                    mark(src, tgt, mname, b, 'calibrate_cost', m, len(buf),
                         fit_s=round(cal_fit, 4), apply_s=round(cal_ap, 4))

                if not is_done(src, tgt, mname, b, 'rethreshold_cost'):
                    t0 = time.time(); bm, bt, bo = best_threshold_two_sided(ybuf, p_buf); sw = time.time() - t0
                    p_or = p_ev if bo == 1 else 1.0 - p_ev
                    p_or_buf = p_buf if bo == 1 else 1.0 - p_buf
                    t0 = time.time(); _ = (p_or >= bt).astype(int); ap = time.time() - t0
                    m = all_metrics(yev, p_or)
                    m.update(metrics_at_threshold(yev, p_or, bt))
                    m.update(operating_points(yev, p_or, ybuf, p_or_buf))
                    mark(src, tgt, mname, b, 'rethreshold_cost', m, len(buf),
                         fit_s=round(sw, 4), apply_s=round(ap, 4))

                if b in CFG['ft_budgets'] and not single:
                    pend = [k for k in CFG['ft_mult'] if not is_done(src, tgt, mname, b, f'finetune_x{k}')]
                    if pend:
                        ftm = copy.deepcopy(src_model); prev = 0
                        for k in CFG['ft_mult']:
                            step = (100 * k if mname in ('rf', 'lgbm') else 30 * k) - prev
                            prev += step
                            t0 = time.time()
                            ftm = finetune(ftm, mname, Xb_s, ybuf, seed, add_trees=step, ft_epochs=step)
                            if k not in pend:
                                continue
                            p_ft = ftm.predict_proba(Xev_s)[:, 1]; fs = round(time.time() - t0, 1)
                            m = all_metrics(yev, p_ft)
                            m.update(operating_points(yev, p_ft, ybuf, ftm.predict_proba(Xb_s)[:, 1]))
                            mark(src, tgt, mname, b, f'finetune_x{k}', m, len(Xs) + len(buf), fit_s=fs)
                        del ftm; gc.collect()
                del Xb_s
            del Xev_s, p_ev; gc.collect()
        del src_model; gc.collect()
    del Xs; gc.collect()

for tgt in CFG['corpora']:
    ev = parts[tgt]['eval']; yev = ev['Label'].values
    for mname in MODELS:
        jobs = [(b, 'buffer_strat', BUF[(tgt, b)], '') for b in CFG['budgets']] +                [(b, 'buffer_diversity', DIV[(tgt, b)], '') for b in CFG['div_budgets']] +                [(CFG['hold_budget'], 'buffer_holdout', HOLD[(tgt, f)], f) for f in FAMS[tgt]]
        for b, s, buf, fam in jobs:
            if is_done('any', tgt, mname, b, s, fam):
                continue
            ybuf = buf['Label'].values
            if len(np.unique(ybuf)) < 2:
                mark('any', tgt, mname, b, s, {'mcc': 0.0}, len(buf), fam=fam)
                continue
            Xb, mb = clean_X(buf, FEATURES); Xev_b, _ = clean_X(ev, FEATURES, medians=mb)
            mdl = make_model(mname, seed, len(Xb))
            t0 = time.time(); mdl.fit(Xb, ybuf); fs = round(time.time() - t0, 1)
            p_ev = mdl.predict_proba(Xev_b)[:, 1]; p_bf = mdl.predict_proba(Xb)[:, 1]
            m = all_metrics(yev, p_ev); m.update(operating_points(yev, p_ev, ybuf, p_bf))
            extra = {}
            if s == 'buffer_holdout':
                thr = threshold_for_fpr_on_buffer(ybuf, p_bf, CFG['fpr_caps'][0])
                hm = (ev['Attack'] == fam).values
                if hm.sum() and not np.isnan(thr):
                    m['tpr_heldout_family'] = float((p_ev[hm] >= thr).mean())
                extra['apply_s'] = round(float(thr), 6) if not np.isnan(thr) else np.nan
            mark('any', tgt, mname, b, s, m, len(buf), fam=fam, fit_s=fs, **extra)
            del mdl, Xb, Xev_b; gc.collect()
    gc.collect()

print('rows recorded:', len(done))

In [ ]:
from scipy.stats import wilcoxon

v5 = pd.read_csv(V5_CSV).drop_duplicates(['source','target','model','budget','strategy','held_family'])
v1 = pd.read_csv(f'{RESULT}/fc_results.csv'); v3 = pd.read_csv(f'{RESULT}/fc_results_v3.csv')
v4 = pd.read_csv(f'{RESULT}/fc_results_v4.csv')

print('=== COST: wall-clock seconds per strategy, reference seed ===')
cost = []
for nm, df, s in [('calibrate', v5, 'calibrate_cost'), ('rethreshold', v5, 'rethreshold_cost'),
                  ('buffer_only', v1, 'buffer_only'), ('augment', v1, 'augment'),
                  ('iw_augment', v4, 'iw_augment'), ('finetune', v4, 'finetune'),
                  ('source fit (reference)', v1, 'in_domain')]:
    g = df[(df.strategy == s)]
    if 'seed' in g.columns: g = g[g.seed == CFG['seed']]
    if not len(g): continue
    ap = g.apply_s.mean() if 'apply_s' in g.columns and g.apply_s.notna().any() else 0.0
    cost.append(dict(strategy=nm, n=len(g), fit_s=g.fit_s.mean(), fit_max=g.fit_s.max(), apply_s=ap))
CT = pd.DataFrame(cost).round(3); print(CT.to_string(index=False)); CT.to_csv(f'{RESULT}/fc_cost.csv', index=False)

print('\n=== OPERATING POINTS: TPR at a capped FPR, threshold set on the buffer ===')
op = v5[v5.strategy.isin(['calibrate_cost','rethreshold_cost','buffer_strat','buffer_diversity'])].copy()
op['strategy'] = op.strategy.str.replace('_cost','').str.replace('buffer_strat','retrain (random)').str.replace('buffer_diversity','retrain (diversity)')
print(op.groupby(['strategy','budget'])[['mcc','tpr_at_fpr01_buffer','fpr_at_fpr01_buffer',
      'tpr_at_fpr001_buffer','tpr_at_fpr01_oracle']].mean().round(3).to_string())
op.groupby(['strategy','budget'])[['mcc','tpr_at_fpr01_buffer','fpr_at_fpr01_buffer','tpr_at_fpr001_buffer','tpr_at_fpr01_oracle']].mean().round(4).reset_index().to_csv(f'{RESULT}/fc_operating_points.csv', index=False)

print('\n=== FINE-TUNE UPDATE SIZE ===')
ft = v5[v5.strategy.str.startswith('finetune_x')].copy()
ft['mult'] = ft.strategy.str.replace('finetune_x','').astype(int)
base = v4[(v4.strategy=='finetune')&(v4.seed==CFG['seed'])][['source','target','model','budget','mcc']].rename(columns={'mcc':'nb13'})
bo = v1[(v1.strategy=='buffer_only')&(v1.seed==CFG['seed'])][['source','target','model','budget','mcc']].rename(columns={'mcc':'buffer_only'})
tab = ft.pivot_table(index=['source','target','model','budget'], columns='mult', values='mcc').reset_index().merge(bo, on=['source','target','model','budget'])
print(tab.groupby('budget')[[1,3,10,'buffer_only']].mean().round(3).to_string())
print('\nper model (update multiplier 1 / 3 / 10 against buffer-only):')
print(tab.groupby('model')[[1,3,10,'buffer_only']].mean().round(3).to_string())
for k in [1,3,10]:
    d = tab['buffer_only'] - tab[k]
    pair = tab.groupby(['source','target'])[['buffer_only',k]].mean()
    print(f'  x{k}: buffer_only leads by {d.mean():+.3f}, wins {int((d>0).sum())}/{len(d)}, pair-level p={wilcoxon(pair.buffer_only,pair[k]).pvalue:.4f}')
tab.round(4).to_csv(f'{RESULT}/fc_finetune_sweep.csv', index=False)

print('\n=== DURABILITY: retrained on a buffer missing one attack family ===')
h = v5[v5.strategy=='buffer_holdout']
full = v5[(v5.strategy=='buffer_strat')&(v5.budget==CFG['hold_budget'])][['target','model','mcc','tpr_at_fpr01_buffer']].rename(columns={'mcc':'mcc_full','tpr_at_fpr01_buffer':'tpr_full'})
H = h.merge(full, on=['target','model'])
print(H.groupby('target')[['mcc','mcc_full','tpr_at_fpr01_buffer','tpr_full','tpr_heldout_family']].mean().round(3).to_string())
print('\nby held-out family:')
print(H.groupby(['target','held_family'])[['mcc','tpr_heldout_family']].mean().round(3).to_string())
print(f"\noverall: MCC {H.mcc.mean():.3f} vs {H.mcc_full.mean():.3f} with the family present; "
      f"recall on the unseen family {H.tpr_heldout_family.mean():.3f} against {H.tpr_full.mean():.3f} overall TPR at 1% FPR")
H.round(4).to_csv(f'{RESULT}/fc_family_holdout.csv', index=False)

print('\n=== RE-TESTS the referee asked for ===')
acq = v3[(v3.strategy.isin(['acq_diversity','acq_uniform']))&(v3.seed==CFG['seed'])].copy()
acq['rule'] = acq.strategy.str.replace('acq_','')
bo_all = v1[(v1.strategy=='buffer_only')&(v1.seed==CFG['seed'])]
print('  diversity against stratified random, unit = target corpus (the rule uses one buffer per target):')
for bud in sorted(acq.budget.unique()):
    g = acq[(acq.rule=='diversity')&(acq.budget==bud)].groupby('target').mcc.mean()
    base_t = bo_all[bo_all.budget==bud].groupby('target').mcc.mean().reindex(g.index).dropna()
    g = g.reindex(base_t.index)
    if len(g) < 3: continue
    p = wilcoxon(g.values, base_t.values).pvalue
    print(f'    budget {bud}: n={len(g)}, gain {(g-base_t).mean():+.3f}, wins {int((g>base_t).sum())}/{len(g)}, p={p:.3f} '
          f'(smallest attainable p at this n is {2/2**len(g):.3f})')
al = v4[(v4.strategy=='al_iterative')&(v4.seed==CFG['seed'])]
print('  iterative AL against stratified random, bootstrap interval on the mean difference:')
for bud in sorted(al.budget.unique()):
    g = al[al.budget==bud].groupby(['target','model']).mcc.mean()
    bt = bo_all[bo_all.budget==bud].groupby(['target','model']).mcc.mean().reindex(g.index).dropna()
    g = g.reindex(bt.index); d = (g-bt).values
    if len(d) < 3: continue
    bs = np.array([np.mean(np.random.default_rng(s).choice(d, len(d))) for s in range(2000)])
    lo, hi = np.percentile(bs, [2.5, 97.5])
    print(f'    budget {bud}: mean diff {d.mean():+.3f}, 95% CI [{lo:+.3f}, {hi:+.3f}], n={len(d)}')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess
os.chdir('/content/drive/MyDrive/drift-conference')

r = subprocess.run(["python", "tools/commit_cell.py",
  "14: cost, operating points at capped FPR, fine-tune update-size sweep, unseen-family durability"],
  capture_output=True, text=True)
print(r.stdout); print(r.stderr)